In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scanpy as sc
from tqdm.auto import tqdm
from wppkg import write_json
from perthub.tokenizer import tokenize_adata_to_hf_dataset

### Step 1: preprocess your adata

- adata.X: normalize_total and log1p

- check and unify smiles; filter low frequency cells

- calculate rdkit2d embeddings; dose and time normalization

- select top-k hvgs if needed

- split train, valid, and test datasets

- add adata.obs["sample_indices"]

In [3]:
adata = sc.read_h5ad("../../data/sciplex3_biolord.h5ad")

# TODO: feel free to add preprocessing if it's required.
...

# add adata.obs["sample_indices"]
adata.obs["sample_indices"] = np.arange(len(adata))

### Step 2: construct `attributes_map`

- `ordered_attributes_map`

- `categorical_attributes_map`

- `n_samples` 

In [4]:
from collections import defaultdict

attributes_map = defaultdict(dict)

# add `categorical_attributes_map`
categorical_attributes_a2d = {"cell_type": "cell_type"}  # anndata: dataset
for attr_a, attr_d in tqdm(categorical_attributes_a2d.items(), desc="building categorical_attributes_map"):
    attributes_map["categorical_attributes_map"][attr_d] = {cat: i for i, cat in enumerate(adata.obs[attr_a].unique())}

# add `ordered_attributes_map`
ordered_attributes_a2d = {"rdkit2d_dose": "rdkit2d_dose"}  # anndata: dataset
for attr_a, attr_d in tqdm(ordered_attributes_a2d.items(), desc="building ordered_attributes_map"):
    attributes_map["ordered_attributes_map"][attr_d] = adata.obsm[attr_a].shape[-1]

# add `n_samples`
unknown_attributes_a2d = {"sample_indices": "sample_indices"}
# Actually, n_samples just needs to count the total cells from the training and validation sets.
attributes_map["n_samples"] = len(adata)  # just save `n_samples`

# save
write_json(attributes_map, "../../data/attributes_map.json")

building categorical_attributes_map:   0%|          | 0/1 [00:00<?, ?it/s]

building ordered_attributes_map:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
attributes_map

defaultdict(dict,
            {'categorical_attributes_map': {'cell_type': {'A549': 0,
               'MCF7': 1,
               'K562': 2}},
             'ordered_attributes_map': {'rdkit2d_dose': 174},
             'n_samples': 354640})

### Step 3: tokenization

In [6]:
# categorical attributes mapping: str -> int
for attr_a, attr_b in tqdm(categorical_attributes_a2d.items(), desc="converting categorical attributes"):
    adata.obs[attr_a] = adata.obs[attr_a].map(attributes_map["categorical_attributes_map"][attr_b])

converting categorical attributes:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# extract train, valid, test
adata_train = adata[adata.obs["split_ood"] == "train"]
adata_valid = adata[adata.obs["split_ood"] == "test"]
# adata_test = adata[adata.obs["split_ood"] == "ood"]

# tokenize
attr_map_dict = {**categorical_attributes_a2d, **ordered_attributes_a2d, **unknown_attributes_a2d}

ds_train = tokenize_adata_to_hf_dataset(
    adata=adata_train,
    attr_map_dict=attr_map_dict,
    x_key="x"
)
ds_train.save_to_disk("../../data/ds_train")

ds_valid = tokenize_adata_to_hf_dataset(
    adata=adata_valid,
    attr_map_dict=attr_map_dict,
    x_key="x"
)
ds_valid.save_to_disk("../../data/ds_valid")

Tokenizing to HF Dataset:   0%|          | 0/16 [00:00<?, ?it/s]

Saving the dataset (0/6 shards):   0%|          | 0/313598 [00:00<?, ? examples/s]

Tokenizing to HF Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/29192 [00:00<?, ? examples/s]

### Step 4: extract arc institute stack embeddings for each celltype

In [8]:
# ! pip install arc-stack

In [ ]:
from wppkg import guess_is_lognorm, reverse_adata_to_raw_counts

# Make sure your adata involves raw counts in adata.X or adata.raw.X
if adata.raw is None:
    if guess_is_lognorm(adata):
        print("It seems that adata.X is log-normalized, try to convert it back to raw counts.")
        try:
            adata = reverse_adata_to_raw_counts(adata)
            print("Successfully converted adata.X back to raw counts, stack will use adata.X as input features by default.")
        except Exception as e:
            raise ValueError("Failed to convert adata.X back to raw counts. Please make sure your adata contains raw counts in either adata.X or adata.raw.X.")

    else:
        print("Stack will use adata.X as input features by default.")
else:
    print("Stack will use adata.raw.X as input features by default.")

2026-06-05 15:40:30 | INFO | Data appears to be log1p normalized (decimals detected, range [0.00, 5.25])


It seems that adata.X is log-normalized, try to convert it back to raw counts.


2026-06-05 15:40:33 | INFO | Data appears to be log1p normalized (decimals detected, range [0.00, 5.25])
2026-06-05 15:40:57 | INFO | Start reversing back to raw counts.


  0%|          | 0/354640 [00:00<?, ?it/s]

2026-06-05 15:41:29 | INFO | Successfully reversed back to raw counts.


Successfully converted adata.X back to raw counts, stack will use adata.X as input features by default.


In [11]:
import os
import torch
from perthub.cell_embed import ArcStackEmbeddingExtractor

stack_extractor = ArcStackEmbeddingExtractor(
    cache_dir="./arc_institute_stack",
    device=torch.device("cuda:0"),
    use_mirror=True
)

celltype_embeddings = []
for ct_idx in tqdm(sorted(adata.obs["cell_type"].unique()), desc="Extracting cell type embeddings with Arc Institute Stack"):
    tmp_path = f".tmp_{ct_idx}.h5ad"
    adata[(adata.obs["cell_type"] == ct_idx) & (adata.obs["control"] == 1)].write_h5ad(tmp_path)  # Cache control cells
    embeddings = stack_extractor(
        adata_path=tmp_path,
        gene_name_col=None,  # Use adata.var_names as gene names
        batch_size=16,
        num_workers=0  # Avoid semaphore error
    )
    celltype_embeddings.append(embeddings.mean(axis=0).reshape(1, -1))  # Get the mean embedding for this cell type

celltype_embeddings = np.concatenate(celltype_embeddings, axis=0)
print("Cell type embeddings shape:", celltype_embeddings.shape)

# Clean up temporary files
for ct_idx in adata.obs["cell_type"].unique():
    tmp_path = f".tmp_{ct_idx}.h5ad"
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

Extracting cell type embeddings with Arc Institute Stack:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-05 15:45:56 | INFO | STACK will: (1) read from adata.raw.X if present, else adata.X; (2) apply log1p only (no normalization); (3) map genes to its 15,012-gene vocabulary.
'organism' column not found, assuming all cells are valid


Extracting embeddings:   0%|          | 0/1 [00:00<?, ?batch/s]

2026-06-05 15:46:21 | INFO | STACK will: (1) read from adata.raw.X if present, else adata.X; (2) apply log1p only (no normalization); (3) map genes to its 15,012-gene vocabulary.
'organism' column not found, assuming all cells are valid


Extracting embeddings:   0%|          | 0/2 [00:00<?, ?batch/s]

2026-06-05 15:46:38 | INFO | STACK will: (1) read from adata.raw.X if present, else adata.X; (2) apply log1p only (no normalization); (3) map genes to its 15,012-gene vocabulary.
'organism' column not found, assuming all cells are valid


Extracting embeddings:   0%|          | 0/1 [00:00<?, ?batch/s]

Cell type embeddings shape: (3, 1600)


In [12]:
np.save("../../data/stack-embed.npy", celltype_embeddings)